# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following best practices for data exploration and processing.

### Dataset Source
The dataset source (Croissant schema) is accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This will give an overview of the dataset's content and purpose.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m\n\n{metadata.description}\n")

## 2. Data Overview
Let's inspect available record sets and their fields. Below, we print each record set's `@id`, its name, and the fields (with `@id`s) it contains.

*Note: For this dataset, the Croissant schema may include one or more record sets. All references are by their `@id`s.*

In [ ]:
# Helper: get all record sets and their fields by @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the schema.")
else:
    for recset in record_sets:
        print(f"\nRecord Set: {recset['@id']}")
        print(f"  Name: {recset.get('name', '[No name]')}")
        print(f"  Description: {recset.get('description', '[No description]')}")
        fields = recset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        if fields:
            for fld in fields:
                if isinstance(fld, dict):
                    print(f"    - {fld.get('@id','[No id]')} (name: {fld.get('name','[No name]')})")
                else:
                    print(f"    - {fld}")
        else:
            print("    [No fields listed]")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. Use the record set and field `@id`s identified above.

*Note: Adjust the list of record sets' `@id`s as found above. Each DataFrame is indexed by record set `@id`.*

In [ ]:
# Collect all record set @id's
rs_ids = [recset['@id'] for recset in record_sets] if record_sets else []
dataframes = {}
for rs_id in rs_ids:
    records_iter = dataset.records(record_set=rs_id)
    records_list = list(records_iter)
    if records_list:
        dataframes[rs_id] = pd.DataFrame(records_list)
        print(f"Loaded DataFrame for record set: {rs_id} (rows: {len(dataframes[rs_id])})")
    else:
        print(f"No data records found for record set: {rs_id}")

# Preview first record set's columns
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for record set {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's explore one of the record sets by selecting a numeric field for operations like filtering, normalization, and grouping.

*Replace the `<numeric_field_id>` and `<group_field_id>` below with an actual field `@id` from the available columns.*

In [ ]:
import numpy as np

# Select the record set and fields to analyze
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Automatically select a numeric field (float or int)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to auto-convert columns with numeric-like names
        potential_numeric = [col for col in df.columns if 'log_likelihood' in col.lower() or 'coef' in col.lower() or 'value' in col.lower() or 'p_' in col.lower() or 'std' in col.lower() or 'se' in col.lower()]
        for col in potential_numeric:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        # Set filtering threshold
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: (showing top 5)")
        display(filtered_df.head())
        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        # Try grouping by a likely categorical field (first non-numeric field)
        non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric field found to group by.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No record set DataFrames available for EDA.")

## 5. Visualization
Let's visualize the distribution of a selected numeric field and its relationship with a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution and boxplot by group if possible
if dataframes and numeric_fields and len(filtered_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- We loaded and explored the FAIR^2 dataset using `mlcroissant` and Pandas.
- We identified record sets and fields via their `@id`s, and loaded records dynamically.
- Simple EDA and normalization were performed on a numeric field, and results were visualized.
- This workflow can be extended for further analysis, including advanced statistical modeling and custom visualizations specific to the survey or regression outputs included in the dataset.

***
_All operations and entity references followed the Croissant specification, ensuring reproducibility and FAIR data principles._